# Framework for using SEAS5, ARPEGE and AROME forecast

It fetches forecasts, and uses it as the basis of decision for irrigation of a parcel

# FULL PIPELINE: from raw data to irrigation recommendation

In [ ]:
# Assume Block 0 data is downloaded, Block 2 daily ERA5 is processed.
# This is what runs daily (or on-demand) for one parcel.

from datetime import date
from pathlib import Path

from irrigator.config import load_parcel_config, load_region_config

cfg = load_region_config("configs/dordogne.yaml")
parcel = load_parcel_config("configs/parcels/example.yaml")

## ===========================================================================
## BLOCK 1 — Static layers (run once per parcel, cache the result)
## ===========================================================================

In [ ]:
from irrigator.static_layers.soil import get_soil_profile
from irrigator.static_layers.terrain import get_era5_elevation, get_terrain_params

soil = get_soil_profile(cfg, parcel)
terrain = get_terrain_params(cfg, parcel)
terrain.era5_elevation_m = get_era5_elevation(cfg, parcel)

## ===========================================================================
## BLOCK 2 — Historical atmospheric forcing (run once, load cached daily)
## ===========================================================================

In [ ]:
from irrigator.atmospheric.era5_processor import load_daily
from irrigator.atmospheric.forcing import extract_parcel_forcing

era5_daily = load_daily(cfg)  # loads processed/dordogne/atmospheric/era5_daily.nc
forcing = extract_parcel_forcing(era5_daily, parcel, terrain)
# forcing is a DailyForcing: all historical days, downscaled to parcel

## ===========================================================================
## BLOCKS 3+4 — Historical simulation up to yesterday
## ===========================================================================

In [ ]:
from irrigator.crop.phenology import CropParams
from irrigator.water_balance.bucket_model import run_simulation

crop_params = CropParams.from_config(parcel.crop)

# Run from Jan 1 to yesterday — gives us the current soil state
today = date(2026, 7, 15)
historical_forcing = forcing.slice(date(2026, 1, 1), today)

states = run_simulation(
    forcing=historical_forcing,
    parcel=parcel,
    terrain=terrain,
    soil=soil,
    crop_params=crop_params,
)

current_state = states[-1]  # today's water balance state
# current_state.depletion = 85 mm, current_state.stress_coeff = 0.92, etc.

## ===========================================================================
## BLOCK 5a — Short-term forecast (AROME 48h + ARPEGE 96h)
## ===========================================================================

In [ ]:
# Step 1: Fetch the latest forecasts (raw GRIB/NetCDF)
from irrigator.ingestion.meteofrance_client import fetch_latest_forecasts, open_forecast

forecast_files = fetch_latest_forecasts(cfg)
# forecast_files = {"arome": Path("data/raw/.../arome_2026-07-15_00z.grib2"),
#                   "arpege": Path("data/raw/.../arpege_2026-07-15_00z.grib2")}

# Step 2: Open and standardize to ERA5-Land variable names/units
from irrigator.forecasts.short_term import standardize_forecast_to_era5_format

arome_raw = open_forecast(forecast_files["arome"])
arome_daily = standardize_forecast_to_era5_format(arome_raw, source="arome")
# arome_daily is now an xr.Dataset with t_mean, t_min, t_max, precip_mm, etc.
# Same variable names as era5_daily — just covering the next 48h

arpege_raw = open_forecast(forecast_files["arpege"])
arpege_daily = standardize_forecast_to_era5_format(arpege_raw, source="arpege")

# Step 3: Blend — AROME for 0-48h, ARPEGE for 48-96h
import xarray as xr

time_dim = "valid_time"
arome_end = arome_daily[time_dim].values[-1]
arpege_extension = arpege_daily.sel({time_dim: arpege_daily[time_dim] > arome_end})
forecast_gridded = xr.concat([arome_daily, arpege_extension], dim=time_dim)

# Step 4: Downscale to parcel — SAME function as Block 2, no duplication
forecast_forcing = extract_parcel_forcing(forecast_gridded, parcel, terrain)
# forecast_forcing is a DailyForcing for the next 4 days, at parcel level

# Step 5: Run forward water balance under forecast (no irrigation)
from irrigator.forecasts.short_term import run_forward_balance, will_stress_occur

forecast_states = run_forward_balance(
    current_state=current_state,
    forecast_forcing=forecast_forcing,
    parcel=parcel,
    terrain=terrain,
    soil=soil,
    crop_params=crop_params,
)

stress_info = will_stress_occur(forecast_states)
# stress_info = {"stress_expected": True, "days_until_stress": 2,
#                "worst_ks": 0.65, "total_forecast_precip": 3.2}

## ===========================================================================
## BLOCK 5b — Seasonal outlook (SEAS5, run monthly or at season start)
## ===========================================================================

In [ ]:
# Step 1: Fetch SEAS5 anomaly from CDS
from irrigator.ingestion.cds_client import fetch_seas5, open_seas5

fetch_seas5(cfg, year=2026, month=7)  # July initialization
seas5_anom = open_seas5(cfg, year=2026, month=7)

# Step 2: Bias-correct SEAS5 (delta method: ERA5 clim + SEAS5 anomaly)
from irrigator.forecasts.seas5_processor import (
    build_era5_monthly_climatology,
    correct_seas5_monthly,
)

era5_clim = build_era5_monthly_climatology(cfg, start_year=1993, end_year=2016)
seas5_corrected = correct_seas5_monthly(seas5_anom, era5_clim, init_month=7)
# seas5_corrected: 51 members × 6 lead months, on ERA5 absolute scale

# Step 3: Analog disaggregation — match at 1°, extract at 0.1°
from irrigator.forecasts.analog_disaggregation import generate_seasonal_scenarios

scenarios = generate_seasonal_scenarios(
    cfg=cfg,
    era5_daily=era5_daily,  # historical ERA5-Land at native resolution
    seas5_corrected=seas5_corrected,
    init_month=7,
    n_leads=4,  # Jul-Oct (rest of growing season)
)
# scenarios: 51 SeasonalScenario objects
# Each contains daily_ds: xr.Dataset at ERA5-Land 0.1° resolution
# NOT yet downscaled to parcel — that happens in the next step

# Step 4: Seasonal outlook (downscales + runs simulation for each member)
from irrigator.decision.seasonal_outlook import compute_seasonal_outlook

outlook = compute_seasonal_outlook(
    scenarios=scenarios,
    parcel=parcel,
    terrain=terrain,  # extract_parcel_forcing uses this for lapse rate
    soil=soil,
    crop_params=crop_params,
)
# outlook.total_irrigation_median = 145 mm
# outlook.total_irrigation_p75 = 195 mm
# outlook.season_description = "Drier than average — plan for significant irrigation"

## ===========================================================================
## BLOCK 6 — Decision: combine short-term + seasonal into recommendation
## ===========================================================================

In [ ]:
from irrigator.decision.output import build_parcel_report, format_sms, save_report
from irrigator.decision.rules import DecisionParams, compute_recommendation

params = DecisionParams.from_config(parcel=parcel)

advice = compute_recommendation(
    current_state=current_state,
    forecast_states=forecast_states,
    crop_params=crop_params,
    params=params,
    last_irrigation_date=date(2026, 7, 8),  # from farmer's records
)

# Build and save the full report
report = build_parcel_report(parcel.id, today, advice, outlook)
save_report(report, Path("output/"))

# Or send as SMS
sms = format_sms(advice)
print(sms)
# [IrriGator] 2026-07-15: IRRIGUER 35mm le 2026-07-16.
# Réserve sol: 92%. Stade: mid.

#### sms/output would look like : 

{
  "parcel_id": "24_bergerac_001",
  "date": "2026-07-15",
  "recommendation": {
    "date": "2026-07-15",
    "irrigate": true,
    "dose_mm": 35.0,
    "recommended_date": "2026-07-16",
    "reason": "Depletion 85/175 mm approaching stress threshold. Forecast rain: 2 mm (discounted). Recommend 35 mm on 2026-07-16 (CRITICAL: flowering window)",
    "confidence": "high",
    "current_depletion_mm": 85.0,
    "current_taw_mm": 175.0,
    "stress_coefficient": 0.920,
    "crop_stage": "mid",
    "forecast_precip_mm": 2.2,
    "days_until_stress": 2
  },
  "seasonal_outlook": {
    "n_ensemble_members": 51,
    "total_irrigation_mm": {
      "p25": 100,
      "median": 145,
      "p75": 195,
      "mean": 152
    },
    "stress_days": {
      "median": 18,
      "p75": 28
    },
    "description": "Drier than average — plan for significant irrigation"
  }
}